# 04 - Análise descritiva dos resultados

Carrega o CSV bruto gerado por `python -m src.pipeline.run_all` e produz:
1. Tabela-resumo com média e desvio das métricas por modelo.
2. Gráfico de barras da métrica principal (AUC OVO).
3. Boxplot da métrica por modelo.
4. Análise por regime (tamanho, número de classes, proporção categórica, missing).
5. Custo computacional vs. desempenho.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path('..').resolve()))

import pandas as pd

from data.load_tabarena import summarize, RECOMMENDED_TASK_IDS
from src.reports.results_table import summary_by_model
from src.reports.plots import (
    bar_metric_by_model,
    boxplot_metric_by_model,
    bar_metric_by_regime,
    time_vs_quality,
)
from src.pipeline.regime import assign_regimes

In [ ]:
raw = pd.read_csv('../results/raw.csv')
raw.head()

## 1. Tabela-resumo por modelo

In [ ]:
summary = summary_by_model(raw)
summary

## 2. Gráfico de barras (AUC OVO)

In [ ]:
fig = bar_metric_by_model(raw, metric='auc_ovo', output_path=Path('../results/figures/bar_auc.png'))
fig

## 3. Boxplot por modelo

In [ ]:
fig = boxplot_metric_by_model(raw, metric='auc_ovo', output_path=Path('../results/figures/box_auc.png'))
fig

## 4. Análise por regime

In [ ]:
metadata = assign_regimes(summarize(RECOMMENDED_TASK_IDS))
metadata

In [ ]:
for col in ['regime_size', 'regime_classes', 'regime_cat_share', 'regime_missing']:
    fig = bar_metric_by_regime(
        raw, metadata, regime_col=col, metric='auc_ovo',
        output_path=Path(f'../results/figures/regime_{col}.png'),
    )
    fig.suptitle('')
    fig.show()

## 5. Custo computacional vs. desempenho

In [ ]:
fig = time_vs_quality(raw, x='total_time_s', y='auc_ovo', output_path=Path('../results/figures/time_vs_quality.png'))
fig